# Sim Flip Chip Distance - Route B - Mesh Profile Without Distance Field

This notebook profiles the big Route B XAO with Gmsh defaults and no
Distance/Threshold refinement field. Use it as the first big-geometry mesh
baseline before trying gsim-like refinement.

Observed result on the saved probe:

| Stage | Time | Output |
| --- | ---: | --- |
| open XAO | `3.44s` | `102657` points, `154051` curves, `51421` surfaces, `4` volumes |
| `generate(1)` | `2.89s` | `154287` 1D elements |
| `generate(2)` | `119.47s` | `4346569` 2D elements, `2147577` nodes |
| `generate(3)` | `>5 min` | interrupted |


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate repository root")
    ROOT = ROOT.parent

RUN_FOLDER = ROOT / "tutorials" / "runs" / "no_inset" / "sim_flip_chip_distance" / "route_b_ignore_port_sheet_probe"
METADATA_DIR = RUN_FOLDER / "metadata" / "semantic_geometry"
XAO_PATH = RUN_FOLDER / "geometry" / "semantic_geometry_route_b.xao"
PHYSICAL_GROUPS_PATH = METADATA_DIR / "04_export_physical_groups.json"

print("run folder:", RUN_FOLDER)
print("XAO exists:", XAO_PATH.is_file(), XAO_PATH)
print("physical groups exists:", PHYSICAL_GROUPS_PATH.is_file(), PHYSICAL_GROUPS_PATH)


In [ ]:
def read_jsonl(path):
    rows = []
    if not path.is_file():
        print("missing", path)
        return rows
    for line in path.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def print_profile(rows):
    for row in rows:
        event = row.get("event")
        seconds = row.get("seconds")
        elapsed = row.get("elapsed_s")
        detail = row.get("counts") or row.get("refinement") or row.get("entities") or ""
        print(f"{event:24s} seconds={seconds!s:>10s} elapsed={elapsed!s:>10s} {detail}")

rows = read_jsonl(METADATA_DIR / "mesh_stage_profile_gmsh_default_no_field.jsonl")
print_profile(rows)


## Optional Rerun

This rerun intentionally uses Gmsh defaults and does not install any mesh
field. `RUN_3D` defaults to `False` because the observed 3D stage exceeded
five minutes.


In [ ]:
RUN_PROFILE = False
RUN_3D = False

if not RUN_PROFILE:
    print("Set RUN_PROFILE = True to rerun the no-field profile.")
else:
    import time
    import gmsh

    def mesh_counts():
        result = {}
        node_tags, _, _ = gmsh.model.mesh.getNodes()
        result["nodes"] = len(node_tags)
        for dim in (1, 2, 3):
            _types, tags, _nodes = gmsh.model.mesh.getElements(dim)
            result[f"elements_{dim}d"] = sum(len(t) for t in tags)
        return result

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)
    try:
        gmsh.open(str(XAO_PATH))
        for dim in (1, 2, 3):
            if dim == 3 and not RUN_3D:
                print("skip generate(3); set RUN_3D = True to run it")
                break
            started = time.perf_counter()
            gmsh.model.mesh.generate(dim)
            print(f"generate({dim})", round(time.perf_counter() - started, 3), mesh_counts())
    finally:
        gmsh.clear()
        gmsh.finalize()
